# utils.py – Documentation and Examples
**Author: Jose Calatayud**

This notebook explains each function in `utils.py` with worked examples.

The module is organised into three sections:
1. **Tokenizers** – four approaches that trade speed vs. linguistic precision
2. **Similarity / distance functions** – all implemented from scratch
3. **Feature extraction** – assembles features from the above building blocks

In [ ]:
from utils import (
    whitespace_tokenizer, regex_tokenizer, clean_tokenizer, char_ngram_tokenizer,
    jaccard_similarity, word_overlap_ratio, cosine_similarity_bow,
    levenshtein_distance, normalized_levenshtein,
    ENGLISH_STOPWORDS, WH_WORDS,
)

# Two representative Quora question pairs
Q1A = "Why do I get easily bored with everything?"
Q2A = "Why do I get bored with things so quickly and easily?"  # duplicate

Q1B = "What is the best programming language for beginners?"
Q2B = "What is the best way to learn machine learning?"       # not duplicate

---
## Section 1 – Tokenizers

Four strategies capture different levels of linguistic structure.

### 1.1 `whitespace_tokenizer`

Splits on whitespace only.  
**Advantage**: zero dependencies, fastest possible.  
**Disadvantage**: punctuation stays attached — `'everything?'` ≠ `'everything'`.

In [ ]:
print('Q1A whitespace:', whitespace_tokenizer(Q1A))
print('Q2A whitespace:', whitespace_tokenizer(Q2A))

### 1.2 `regex_tokenizer`

Extracts `\w+` tokens via regex – strips all punctuation.  
**Advantage**: handles contractions, possessives, mixed punctuation cleanly.  
**Disadvantage**: splits contractions (e.g. `"don't"` → `['don', 't']`).

In [ ]:
print('Q1A regex:', regex_tokenizer(Q1A))
print('Q2A regex:', regex_tokenizer(Q2A))
print()
# Demonstrate punctuation stripping
print('Contraction demo:', regex_tokenizer("I don't know, it's fine."))

### 1.3 `clean_tokenizer`

Regex tokeniser that additionally removes English stopwords.  
**Advantage**: keeps only content words — most informative for semantic similarity.  
**Disadvantage**: can lose signal from questions like `'Who is who?'`.

In [ ]:
print('Q1A clean:', clean_tokenizer(Q1A))
print('Q2A clean:', clean_tokenizer(Q2A))
print()
print('Sample stopwords removed:', sorted(ENGLISH_STOPWORDS)[:10], '...')

### 1.4 `char_ngram_tokenizer`

Generates character trigrams (default `n=3`).  
**Advantage**: robust to typos, spelling variants, and morphological inflections  
(`'bored'` and `'boring'` share `'bor', 'ore'`).  
**Disadvantage**: ignores word boundaries; very long feature vectors.

In [ ]:
sample = "bored quickly"
trigrams = char_ngram_tokenizer(sample)
print(f'Char trigrams of "{sample}":', trigrams)
print()
# Morphological overlap demo
tg_bored  = set(char_ngram_tokenizer('bored'))
tg_boring = set(char_ngram_tokenizer('boring'))
print('Shared trigrams between "bored" and "boring":', tg_bored & tg_boring)

### Tokeniser comparison on the same pair

In [ ]:
print('=== Pair A (duplicate) ===')
for name, fn in [('whitespace', whitespace_tokenizer),
                 ('regex',      regex_tokenizer),
                 ('clean',      clean_tokenizer)]:
    t1, t2 = fn(Q1A), fn(Q2A)
    j = jaccard_similarity(t1, t2)
    print(f'  [{name:12s}] Jaccard = {j:.3f}  |  tokens: {len(t1)} vs {len(t2)}')

print()
print('=== Pair B (non-duplicate) ===')
for name, fn in [('whitespace', whitespace_tokenizer),
                 ('regex',      regex_tokenizer),
                 ('clean',      clean_tokenizer)]:
    t1, t2 = fn(Q1B), fn(Q2B)
    j = jaccard_similarity(t1, t2)
    print(f'  [{name:12s}] Jaccard = {j:.3f}  |  tokens: {len(t1)} vs {len(t2)}')

---
## Section 2 – Similarity / Distance Functions (from scratch)

All functions use only built-in Python; no sklearn or scipy in the computation path.

### 2.1 `jaccard_similarity`

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Symmetric, range [0, 1]. Penalises extra tokens in either direction.

In [ ]:
t1 = regex_tokenizer(Q1A)
t2 = regex_tokenizer(Q2A)
t3 = regex_tokenizer(Q1B)
t4 = regex_tokenizer(Q2B)

print('Jaccard (duplicate pair A):     ', jaccard_similarity(t1, t2))
print('Jaccard (non-duplicate pair B): ', jaccard_similarity(t3, t4))
print('Jaccard (identical texts):      ', jaccard_similarity(t1, t1))
print('Jaccard (empty vs empty):       ', jaccard_similarity([], []))

### 2.2 `word_overlap_ratio`

$$\text{overlap}(A, B) = \frac{|A \cap B|}{\min(|A|, |B|)}$$

Unlike Jaccard, a short question that is fully contained in a longer one gets score 1.0.  
Useful because Quora duplicates often rephrase using more or fewer words.

In [ ]:
# Short question fully inside long question
short = regex_tokenizer('How to learn Python?')
long  = regex_tokenizer('What is the best way to learn Python programming for beginners?')

print('Jaccard (short vs long): ', round(jaccard_similarity(short, long), 3))
print('Overlap  (short vs long):', round(word_overlap_ratio(short, long), 3))
print()
print('Overlap (pair A):', round(word_overlap_ratio(
    clean_tokenizer(Q1A), clean_tokenizer(Q2A)), 3))
print('Overlap (pair B):', round(word_overlap_ratio(
    clean_tokenizer(Q1B), clean_tokenizer(Q2B)), 3))

### 2.3 `cosine_similarity_bow`

$$\cos(A, B) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \cdot \|\vec{B}\|}$$

Uses term frequency counts (not binary). Unlike Jaccard, repeated words contribute more.

Implemented from scratch using dicts — no numpy or sklearn.

In [ ]:
print('BoW cosine (pair A):', round(cosine_similarity_bow(
    regex_tokenizer(Q1A), regex_tokenizer(Q2A)), 3))
print('BoW cosine (pair B):', round(cosine_similarity_bow(
    regex_tokenizer(Q1B), regex_tokenizer(Q2B)), 3))
print()
# Frequency sensitivity demo: repeated keyword bumps similarity
a = ['python', 'python', 'code']
b = ['python', 'tutorial']
print('High overlap  (python×2 vs python+tutorial):', round(cosine_similarity_bow(a, b), 3))
print('Jaccard same pair:                          ', round(jaccard_similarity(a, b), 3))

### 2.4 `levenshtein_distance` and `normalized_levenshtein`

Classic edit distance: minimum number of single-character insertions, deletions,
or substitutions to transform one string into another.

Implemented with O(n) space dynamic programming (only one row needed at a time).
Strings are truncated to 300 characters to bound worst-case runtime.

In [ ]:
# Raw distance
print('levenshtein("kitten", "sitting") =', levenshtein_distance('kitten', 'sitting'))
print('levenshtein("bored",  "boring")  =', levenshtein_distance('bored',  'boring'))
print()
# Normalised distance on the question pairs
print('Normalised Levenshtein (pair A):', round(normalized_levenshtein(Q1A, Q2A), 3))
print('Normalised Levenshtein (pair B):', round(normalized_levenshtein(Q1B, Q2B), 3))
print()
# Sanity checks
print('Identical strings → 0.0:', normalized_levenshtein(Q1A, Q1A))
print('Empty strings     → 0.0:', normalized_levenshtein('', ''))

---
## Section 3 – Feature Extraction

The two feature extraction functions are wrappers that apply the above building
blocks to a full DataFrame.

### 3.1 `extract_simple_features`

Computes **one feature** per pair: cosine similarity between the TF-IDF vectors
of `question1` and `question2`.

The vectorizer must be **fitted on training data** and the same fitted object
must be reused for validation and test (to avoid data leakage).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from utils import extract_simple_features

demo_df = pd.DataFrame({
    'question1': [Q1A, Q1B],
    'question2': [Q2A, Q2B],
    'is_duplicate': [1, 0],
})

# Fit TF-IDF on all questions (normally fitted on training corpus)
all_q = demo_df['question1'].tolist() + demo_df['question2'].tolist()
tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
tfidf.fit(all_q)

X_simple = extract_simple_features(demo_df, tfidf)
print('Shape:', X_simple.shape)   # (2, 1)
print('TF-IDF cosine similarity per pair:')
for i, row in demo_df.iterrows():
    label = 'DUPLICATE' if row['is_duplicate'] else 'NON-DUPLICATE'
    print(f'  [{label}] sim = {X_simple[i, 0]:.4f}')

### 3.2 `extract_improved_features`

Assembles an **18-feature** dense vector for each pair.  
See the function docstring in `utils.py` for the full feature layout.

In [ ]:
from utils import extract_improved_features

tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 3),
    min_df=1, sublinear_tf=True
)
tfidf_char.fit(all_q)

X_imp = extract_improved_features(demo_df, tfidf, tfidf_char)
print('Shape:', X_imp.shape)   # (2, 18)
print()

feature_names = [
    'regex_jaccard', 'regex_overlap', 'regex_bow_cos', 'regex_len_diff',
    'clean_jaccard', 'clean_overlap', 'clean_bow_cos', 'clean_len_diff',
    'char_jaccard',  'char_overlap',  'char_bow_cos',  'char_len_diff',
    'tfidf_word_cos', 'tfidf_char_cos',
    'norm_levenshtein', 'char_len_ratio',
    'same_wh_word', 'both_wh',
]

print(f'{'Feature':<20}  {'Pair A (dup)':>13}  {'Pair B (non-dup)':>17}')
print('-' * 56)
for name, a, b in zip(feature_names, X_imp[0], X_imp[1]):
    print(f'{name:<20}  {a:>13.4f}  {b:>17.4f}')

**Observations from the feature table:**

- `clean_jaccard` and `clean_overlap` are higher for Pair A (actual duplicate)
  because removing stopwords exposes that the content words are very similar.
- `char_jaccard` and `char_bow_cos` are also higher for Pair A, showing that
  even at the character level the questions look more alike.
- `norm_levenshtein` is lower for Pair A (fewer edits needed).
- `same_wh_word` = 1 for Pair B (`'what'`) but = 0 for Pair A (`'why'` vs `'why'` is 1...)  
  — this illustrates that WH-word matching alone is not enough to distinguish duplicates.

The improved model combines all 18 features with a gradient boosting classifier
that can learn non-linear interactions between them.

---
## Section 4 – Design Choices and Limitations

| Choice | Reason |
|--------|--------|
| Four tokenisers instead of one | Captures different kinds of evidence (word-level, content-word-level, subword). Their signals are complementary. |
| Jaccard + overlap coefficient | Jaccard penalises both sides equally; overlap is lenient toward short paraphrases. |
| BoW cosine from scratch | Captures term frequency, unlike binary Jaccard. Validates the TF-IDF cosine computed by sklearn. |
| Levenshtein from scratch | Edit distance detects near-duplicate wording (e.g. singular vs plural, small typos). |
| HistGradientBoosting over LR | Can learn non-linear feature interactions (e.g. `clean_jaccard` matters more when `both_wh` = 0). |

**Limitations**: No semantic understanding — questions about different topics can share
many words. Adding word embeddings (e.g. SentenceBERT) would further improve accuracy.